# WEEK BY WEEK WINNER PREDICTIONS

## Potential name for application: Gamelytics

In [5]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from tensorflow import keras
from tensorflow.keras import layers

In [6]:
print(nfl.import_schedules([2025]))

              game_id  season game_type  week     gameday   weekday gametime  \
6991  2025_01_DAL_PHI    2025       REG     1  2025-09-04  Thursday    20:20   
6992   2025_01_KC_LAC    2025       REG     1  2025-09-05    Friday    20:00   
6993   2025_01_TB_ATL    2025       REG     1  2025-09-07    Sunday    13:00   
6994  2025_01_CIN_CLE    2025       REG     1  2025-09-07    Sunday    13:00   
6995  2025_01_MIA_IND    2025       REG     1  2025-09-07    Sunday    13:00   
...               ...     ...       ...   ...         ...       ...      ...   
7258  2025_18_DAL_NYG    2025       REG    18  2026-01-04    Sunday    13:00   
7259  2025_18_WAS_PHI    2025       REG    18  2026-01-04    Sunday    13:00   
7260  2025_18_BAL_PIT    2025       REG    18  2026-01-04    Sunday    13:00   
7261   2025_18_SEA_SF    2025       REG    18  2026-01-04    Sunday    13:00   
7262   2025_18_CAR_TB    2025       REG    18  2026-01-04    Sunday    13:00   

     away_team  away_score home_team  .

## 1. Load the schedule data

In [7]:
train_seasons = list(range(2020, 2025))

predict_season = 2025

sched = nfl.import_schedules(train_seasons)

sched_pred = nfl.import_schedules([predict_season])

## 2. Compute team/season features

In [8]:
team_stats = []

for season in train_seasons:
    # get games from this season
    season_games = sched[sched['season'] == season]

    # get all teams that played this season
    all_teams = list(season_games['home_team'].unique()) + list(season_games['away_team'].unique())
    all_teams = list(set(all_teams))  # remove duplicates

    for team in all_teams:
        # games where this team was home or away
        home_games = season_games[season_games['home_team'] == team]
        away_games = season_games[season_games['away_team'] == team]

        # total points scored and allowed
        points_for = home_games['home_score'].sum() + away_games['away_score'].sum()
        points_against = home_games['away_score'].sum() + away_games['home_score'].sum()

        # count wins and losses
        home_wins = (home_games['home_score'] > home_games['away_score']).sum()
        away_wins = (away_games['away_score'] > away_games['home_score']).sum()
        wins = int(home_wins + away_wins)

        home_losses = (home_games['home_score'] < home_games['away_score']).sum()
        away_losses = (away_games['away_score'] < away_games['home_score']).sum()
        losses = int(home_losses + away_losses)

        games_played = len(home_games) + len(away_games)

        # save results
        team_stats.append({
            'season': season,
            'team': team,
            'points_for': points_for,
            'points_against': points_against,
            'wins': wins,
            'losses': losses,
            'games_played': games_played
        })

# make into dataframe
team_feats = pd.DataFrame(team_stats)

# averages
team_feats['avg_points_for'] = team_feats['points_for'] / team_feats['games_played'].replace(0, 1)
team_feats['avg_points_against'] = team_feats['points_against'] / team_feats['games_played'].replace(0, 1)